In [1]:
import torch
import numpy as np
import torch

In [2]:
ckpt = torch.load('./models/best.pt', map_location='cpu')

print(len(ckpt))
for k, v in ckpt.items():
    print(k)
    if 'scaling' in k:
        print(v)


67
model.0.conv.weight
model.0.bn.weight
model.0.bn.bias
model.0.bn.running_mean
model.0.bn.running_var
model.0.bn.num_batches_tracked
model.0.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5859, dtype=torch.float16)
model.2.conv.weight
model.2.bn.weight
model.2.bn.bias
model.2.bn.running_mean
model.2.bn.running_var
model.2.bn.num_batches_tracked
model.2.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5859, dtype=torch.float16)
model.4.conv.weight
model.4.bn.weight
model.4.bn.bias
model.4.bn.running_mean
model.4.bn.running_var
model.4.bn.num_batches_tracked
model.4.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5859, dtype=torch.float16)
model.6.conv.weight
model.6.bn.weight
model.6.bn.bias
model.6.bn.running_mean
model.6.bn.running_var
model.6.bn.num_batches_tracked
model.6.act.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value
tensor(2.5859, dtype=torch.fl

In [3]:
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.util.cleanup import cleanup as qonnx_cleanup

In [4]:
model_file_path = "./models/best.qonnx.onnx"
qonnx_cleanup(model_file_path, out_file=model_file_path)

In [5]:
model = ModelWrapper(model_file_path)

In [6]:
import torch

from finn.util.visualization import showInNetron
from finn.util.pytorch import ToTensor
from brevitas.export import export_qonnx
from qonnx.transformation.merge_onnx_models import MergeONNXModels
from qonnx.core.datatype import DataType
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

build_dir = "./"

## Tidyup Transformation

In [7]:
model = ModelWrapper(model_file_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/yolov1-tiny_tidy.onnx")

In [8]:
showInNetron(build_dir+"/yolov1-tiny_tidy.onnx")

Serving './/yolov1-tiny_tidy.onnx' at http://0.0.0.0:8081


In [10]:
model = ModelWrapper(build_dir+"/yolov1-tiny_tidy.onnx")
global_inp_name = model.graph.input[0].name
ishape = model.get_tensor_shape(global_inp_name)

# preprocessing: torchvision's ToTensor divides uint8 inputs by 255
totensor_pyt = ToTensor()
chkpt_preproc_name = build_dir+"/yolov1-tiny_preproc.onnx"
export_qonnx(totensor_pyt, torch.randn(ishape), chkpt_preproc_name)
qonnx_cleanup(chkpt_preproc_name, out_file=chkpt_preproc_name)
pre_model = ModelWrapper(chkpt_preproc_name)
pre_model = pre_model.transform(ConvertQONNXtoFINN())

# join preprocessing and core model
model = model.transform(MergeONNXModels(pre_model))
# add input quantization annotation: UINT8 for all BNN-PYNQ models
global_inp_name = model.graph.input[0].name
model.set_tensor_datatype(global_inp_name, DataType["UINT8"])

/home/hsbi/Documents/Workspace/FINN/finn/deps/qonnx/src/qonnx/transformation/merge_onnx_models.py:70: UserWarning: [MergeONNXModels] opsets for models to merge differ: 14 vs 17, output model will use opset 17
  warnings.warn(
/home/hsbi/Documents/Workspace/FINN/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [11]:
from qonnx.transformation.infer_datatypes import InferDataTypes
chkpt_name = build_dir+"/yolov1-tiny_preproc_tidy.onnx"

# tidy-up again
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(InferDataTypes())
model = model.transform(RemoveStaticGraphInputs())
model.save(chkpt_name)

In [12]:
showInNetron(build_dir+"/yolov1-tiny_preproc_tidy.onnx", port=2222)

Serving './/yolov1-tiny_preproc_tidy.onnx' at http://0.0.0.0:2222


## Streamlining

In [12]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

In [14]:
model = ModelWrapper(build_dir + "/yolov1-tiny_preproc_tidy.onnx")

In [13]:
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(build_dir + "/yolov1-tiny_streamlined.onnx")

In [17]:
showInNetron(build_dir+"/yolov1-tiny_streamlined.onnx", port=2222)

Serving './/yolov1-tiny_streamlined.onnx' at http://0.0.0.0:2222


In [14]:
from finn.util.basic import pynq_part_map

# change this if you have a different PYNQ board, see list above
pynq_board = "Pynq-Z1"
fpga_part = pynq_part_map[pynq_board]
target_clk_ns = 10

## Conversion to HW layers

In [15]:
import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts

In [16]:
model = ModelWrapper(build_dir + "/yolov1-tiny_streamlined.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(to_hw.InferStreamingMaxPool())
model = model.transform(absorb.AbsorbConsecutiveTransposes())
model = model.transform(RemoveCNVtoFCFlatten())
parent_model = model.transform(CreateDataflowPartition())

parent_model.save(build_dir + "/yolov1-tiny_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/yolov1-tiny_dataflow_model.onnx")

In [22]:
showInNetron(build_dir + "/yolov1-tiny_dataflow_parent.onnx", port=2222)

Serving './/yolov1-tiny_dataflow_parent.onnx' at http://0.0.0.0:2222


In [24]:
showInNetron(build_dir + "/yolov1-tiny_dataflow_model.onnx", port=2222)

Serving './/yolov1-tiny_dataflow_model.onnx' at http://0.0.0.0:2222


## Folding: Adjusting the Parallelism

In [17]:
model = ModelWrapper(build_dir + "/yolov1-tiny_dataflow_model.onnx")
fc_layers = model.get_nodes_by_op_type("MVAU_hls")

In [18]:
# each tuple is (PE, SIMD, in_fifo_depth) for a layer
folding = [
    (8, 9, [8]),
    (4, 4, [8]),
    (4, 4, [8]),
    (2, 2, [8]),
    (2, 2, [8]),
    (2, 2, [4]),
    (4, 4, [4]),
    (4, 4, [8]),
    (4, 4, [8]),
    (1, 8, [8]),
]

for fcl, (pe, simd, ififodepth) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr("PE", pe)
    fcl_inst.set_nodeattr("SIMD", simd)
    fcl_inst.set_nodeattr("inFIFODepths", ififodepth)

In [19]:
fc_layers = model.get_nodes_by_op_type("ConvolutionInputGenerator_rtl")

In [20]:
folding = [
    (3),
    (2),
    (2),
    (2),
    (2),
    (2),
    (2),
    (2),
    (2),
]
for fcl, (simd) in zip(fc_layers, folding):
    fcl_inst = getCustomOp(fcl)
    fcl_inst.set_nodeattr("SIMD", simd)

In [21]:
model = model.transform(GiveUniqueNodeNames())
model.save(build_dir + "/yolov1-tiny_folded.onnx")

In [31]:
showInNetron(build_dir + "/yolov1-tiny_folded.onnx", port=2222)

Serving './/yolov1-tiny_folded.onnx' at http://0.0.0.0:2222


## Estimate Reports

In [ ]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_config

In [45]:
model_file = build_dir + "/yolov1-tiny_folded.onnx"

In [42]:
estimate_output_dir = "output_estimates"

In [44]:
cfg_estimates = build.DataflowBuildConfig(
    output_dir          = estimate_output_dir,
    target_fps          = 15,
    synth_clk_period_ns = 10.0,
    fpga_part           = fpga_part,
    steps               = ["step_generate_estimate_reports"],
    generate_outputs=[
        build_config.DataflowOutputType.ESTIMATE_REPORTS,
    ]
)

In [47]:
%%time
build.build_dataflow_cfg(model_file, cfg_estimates)

Running step: step_generate_estimate_reports [1/1]
/home/suraj-karki/Documents/My_Workspace/Coding_Works/Master_Project/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_0 can be lower than
             actual latency!
  warnings.warn(
/home/suraj-karki/Documents/My_Workspace/Coding_Works/Master_Project/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_1 can be lower than
             actual latency!
  warnings.warn(
/home/suraj-karki/Documents/My_Workspace/Coding_Works/Master_Project/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer StreamingMaxPool_hls_2 can be lower than
             actual latency!
  warnings.warn(
/home/suraj-karki/Documents/My_Workspace/Coding_Works/Master_Project/finn/src/finn/custom_op/fpgadataflow/streamingmaxpool.py:139: UserWarning: Estimated latency for layer 

Building dataflow accelerator from .//yolov1-tiny_folded.onnx
Intermediate outputs will be generated in /tmp/finn_dev_suraj-karki
Final outputs will be generated in output_estimates
Build log is at output_estimates/build_dataflow.log
Running step: step_generate_estimate_reports [1/1]
Completed successfully
CPU times: user 444 ms, sys: 24.7 ms, total: 469 ms
Wall time: 480 ms


0

In [48]:
! ls {estimate_output_dir}

intermediate_models  report  time_per_step.json


In [49]:
! ls {estimate_output_dir}/report

estimate_layer_config_alternatives.json  estimate_network_performance.json
estimate_layer_cycles.json		 op_and_param_counts.json
estimate_layer_resources.json


In [50]:
! cat {estimate_output_dir}/report/estimate_network_performance.json

{
  "critical_path_cycles": 18572706,
  "max_cycles": 3115008,
  "max_cycles_node_name": "MVAU_hls_3",
  "estimated_throughput_fps": 32.102646285338594,
  "estimated_latency_ns": 185727060.0
}

In [51]:
import json
def read_json_dict(filename):
    with open(filename, "r") as f:
        ret = json.load(f)
    return ret

In [52]:
read_json_dict(estimate_output_dir + "/report/estimate_layer_resources.json")

{'FMPadding_rtl_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'ConvolutionInputGenerator_rtl_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 852,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'MVAU_hls_0': {'BRAM_18K': 16,
  'BRAM_efficiency': 0.005859375,
  'LUT': 9214,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'StreamingMaxPool_hls_0': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'FMPadding_rtl_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'ConvolutionInputGenerator_rtl_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 660,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'MVAU_hls_1': {'BRAM_18K': 2,
  'BRAM_efficiency': 0.0625,
  'LUT': 1034,
  'URAM': 0,
  'URAM_efficiency': 1,
  'DSP': 0},
 'StreamingMaxPool_hls_1': {'BRAM_18K': 0,
  'BRAM_efficiency': 1,
  'LUT': 0,
  'URAM': 0,

In [33]:
from finn.transformation.fpgadataflow.make_zynq_proj import ZynqBuild

model = ModelWrapper(build_dir+"/yolov1-tiny_folded.onnx")
model = model.transform(ZynqBuild(platform = pynq_board, period_ns = target_clk_ns))

/mnt/c/Users/Leica-Admin/Documents/Workspace/finn/src/finn/transformation/fpgadataflow/floorplan.py:107: UserWarning: 56 nodes have no entry in the provided floorplan, SLR was set to -1
  warnings.warn(
/mnt/c/Users/Leica-Admin/Documents/Workspace/finn/src/finn/transformation/fpgadataflow/insert_fifo.py:237: UserWarning: Input FIFO for IODMA_hls_0_out0 has depth 2 and won't
                        be created. This may cause RTL simulation issues.
                        
  warnings.warn(
/mnt/c/Users/Leica-Admin/Documents/Workspace/finn/src/finn/transformation/fpgadataflow/insert_fifo.py:299: UserWarning: Output FIFO for MVAU_hls_9_out0 has depth 2 and won't
                        be created. This may cause RTL simulation issues.
                        
  warnings.warn(
/mnt/c/Users/Leica-Admin/Documents/Workspace/finn/src/finn/transformation/fpgadataflow/create_stitched_ip.py:290: UserWarning: First node is not StreamingFIFO or IODMA.
                You may experience incorrect sti

In [34]:
from finn.transformation.fpgadataflow.make_pynq_driver import MakePYNQDriver
model = model.transform(MakePYNQDriver("zynq-iodma"))

In [36]:
model.save(build_dir + "/yolov1-tiny_synth.onnx")
showInNetron(build_dir + "/yolov1-tiny_synth.onnx", port=2222)

Serving './/yolov1-tiny_synth.onnx' at http://0.0.0.0:2222


In [37]:
model = ModelWrapper(build_dir + "/yolov1-tiny_synth.onnx")
model.model.metadata_props

[key: "floorplan_json"
value: "/tmp/finn_dev_surajkarki/vitis_floorplan_6esqdal4/floorplan.json"
, key: "vivado_pynq_proj"
value: "/tmp/finn_dev_surajkarki/vivado_zynq_proj_quwkwo24"
, key: "bitfile"
value: "/tmp/finn_dev_surajkarki/vivado_zynq_proj_quwkwo24/resizer.bit"
, key: "hw_handoff"
value: "/tmp/finn_dev_surajkarki/vivado_zynq_proj_quwkwo24/resizer.hwh"
, key: "vivado_synth_rpt"
value: "/tmp/finn_dev_surajkarki/vivado_zynq_proj_quwkwo24/synth_report.xml"
, key: "platform"
value: "zynq-iodma"
, key: "pynq_driver_dir"
value: "/tmp/finn_dev_surajkarki/pynq_driver_kzf5_12z"
]

In [38]:
! ls {model.get_metadata_prop("vivado_pynq_proj")}

finn_zynq_link.cache	      finn_zynq_link.srcs  resizer.hwh
finn_zynq_link.gen	      finn_zynq_link.xpr   synth_project.sh
finn_zynq_link.hw	      ip_config.tcl	   synth_report.xml
finn_zynq_link.ip_user_files  NA		   vivado.jou
finn_zynq_link.runs	      resizer.bit	   vivado.log


In [39]:
build_dir

'./'

In [41]:
from shutil import copy
from distutils.dir_util import copy_tree
from finn.util.basic import make_build_dir

# create directory for deployment files
deployment_dir = make_build_dir(prefix="pynq_deployment_")
model.set_metadata_prop("pynq_deployment_dir", deployment_dir)

# get and copy necessary files
# .bit and .hwh file
bitfile = model.get_metadata_prop("bitfile")
hwh_file = model.get_metadata_prop("hw_handoff")
deploy_files = [bitfile, hwh_file]

for dfile in deploy_files:
    if dfile is not None:
        copy(dfile, deployment_dir)

# driver.py and python libraries
pynq_driver_dir = model.get_metadata_prop("pynq_driver_dir")
copy_tree(pynq_driver_dir, deployment_dir)

copying /tmp/finn_dev_surajkarki/pynq_driver_kzf5_12z/driver.py -> /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq
copying /tmp/finn_dev_surajkarki/pynq_driver_kzf5_12z/driver_base.py -> /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq
creating /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/finn
creating /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/finn/util
copying /tmp/finn_dev_surajkarki/pynq_driver_kzf5_12z/finn/util/data_packing.py -> /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/finn/util
copying /tmp/finn_dev_surajkarki/pynq_driver_kzf5_12z/finn/util/__init__.py -> /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/finn/util
copying /tmp/finn_dev_surajkarki/pynq_driver_kzf5_12z/validate.py -> /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq
creating /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/runtime_weights
creating /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/qonnx
creating /tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/qonnx/util
copying /tmp/finn

['/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/driver.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/driver_base.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/finn/util/data_packing.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/finn/util/__init__.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/validate.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/qonnx/util/basic.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/qonnx/util/__init__.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/qonnx/core/datatype.py',
 '/tmp/finn_dev_surajkarki/pynq_deployment_jihjeubq/qonnx/core/__init__.py']

In [43]:
! ls {deployment_dir}

driver_base.py	finn   resizer.bit  runtime_weights
driver.py	qonnx  resizer.hwh  validate.py


In [44]:
from shutil import make_archive
make_archive('deploy-on-pynq-z1-tinyyolov3', 'zip', deployment_dir)

'/mnt/c/Users/Leica-Admin/Documents/Workspace/finn/notebooks/experiments/FPGA-acceleration-for-YOLO-based-object-detection/Tiny_YOLOv3/deploy-on-pynq-z1-tinyyolov3.zip'

In [42]:
!ls /tmp/finn_dev_surajkarki/dataflow_partition_5zihu1vt

partition_0.onnx


In [ ]:
make_archive('deploy-on-pynq-z1-tinyyolov3_all', 'zip', "/tmp/finn_dev_surajkarki/")

## Available Resource Utilization Reports

- After synthesis you will find out-of-context synthesis resource utilization inside output directory of finn build.
- output_ipstitch_ooc_rtlsm/report

## Power Utilization Report
- After the implementation, you will be able to see power consumption on vivado stitched ip.